# Paper — 00b: NMS Threshold Sweep

Loads the pre-NMS shapefiles (saved alongside the NMS ones) and re-applies NMS
in memory at different thresholds — no GPU or re-inference needed.

Orientations are computed once per model (expensive), then the NMS keep-mask
is applied cheaply for each threshold.

**Produces:**
- `figures_paper/nms_counts.pdf` — detection count vs. threshold per model
- `figures_paper/nms_histograms.pdf` — orientation histograms across thresholds

In [ ]:
import sys
sys.path.insert(0, "/scratch/users/cayleigh/YOLOv8-BeyondEarth/src")

import typing_extensions
if not hasattr(typing_extensions, "TypeIs"):
    typing_extensions.TypeIs = typing_extensions.TypeGuard

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from pathlib import Path
from shapely import segmentize
from tqdm import tqdm
from lsnms import nms

from rastertools_BOULDERING import metadata as raster_metadata
from shptools_BOULDERING.geometry import fitEllipse
from shptools_BOULDERING.geomorph import boulder_row

In [ ]:
work_dir  = Path.home() / "tmp" / "YOLOv8BeyondEarth"
in_raster = Path("/scratch/users/cayleigh/test_raster/M1221383405.tif")
prieur_test_dir = Path("/scratch/users/cayleigh/Apr2023-Mars-Moon-Earth-mask-5px/preprocessing/test")
gt_tile_ids = ["1386", "1503", "2054", "2277", "2508"]

# Pre-NMS shapefiles — saved alongside the nms ones by get_sliced_prediction_SAM2
# (no -nms suffix = pre-NMS, has -nms suffix = post-NMS)
MODELS = {
    "YOLOv8":          (work_dir / "exp_yolo_256",           "*-downscaled-mask.shp"),
    "SAM2 zero-shot":  (work_dir / "exp_sam2_256",           "*-downscaled-mask.shp"),
    "SAM2 fine-tuned": (work_dir / "exp_sam2_finetuned_256", "*-downscaled-mask.shp"),
    "SAM2-auto":       (work_dir / "exp_sam2_auto_256",      "*-mask.shp"),
}

NMS_THRESHOLDS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]

res             = raster_metadata.get_resolution(in_raster)[0]
AREAL_THRESHOLD = (res ** 2) * (4.74 ** 2)
AR_MIN, AR_MAX  = 1.2, 2.0
BINS            = np.linspace(0, 180, 37)

OUT_DIR = Path("figures_paper"); OUT_DIR.mkdir(exist_ok=True)
plt.rcParams.update({"font.size": 8, "axes.titlesize": 8, "figure.dpi": 150})
print(f"res={res:.4f} m/px   areal_threshold={AREAL_THRESHOLD:.2f} m²")

In [ ]:
def run_pipeline(poly, res):
    """segmentize → fitEllipse → boulder_row. Returns (angle180, ar) or None."""
    row_seg = pd.Series({"geometry": segmentize(poly, res)})
    try:
        ellipse_poly, _, _, _ = fitEllipse(row_seg)
    except Exception:
        return None
    try:
        mrr_row = pd.Series({"geometry": ellipse_poly.minimum_rotated_rectangle})
        _, _, long_ax, short_ax, _, _, _, angle180 = boulder_row(mrr_row)
    except Exception:
        return None
    if short_ax < 1e-6:
        return None
    return angle180, long_ax / short_ax


def apply_nms_to_gdf(gdf, iou_threshold):
    """Apply NMS in memory using polygon bounds as bboxes. Returns boolean keep-mask."""
    bounds = np.array([geom.bounds for geom in gdf.geometry])  # (N, 4): minx miny maxx maxy
    # lsnms expects [x1, y1, x2, y2] — bounds is already in that order
    boxes  = bounds.astype(np.float32)
    scores = gdf["score"].values.astype(np.float32)
    keep   = nms(boxes=boxes, scores=scores, iou_threshold=iou_threshold,
                 class_ids=None, rtree_leaf_size=32)
    mask = np.zeros(len(gdf), dtype=bool)
    mask[keep] = True
    return mask

In [ ]:
# GT orientations — flat reference line for histogram overlays
gt_angles = []
for tile_id in gt_tile_ids:
    shp = prieur_test_dir / "labels" / f"M1221383405_{tile_id}_mask.shp"
    if not shp.exists(): continue
    for geom in gpd.read_file(shp).geometry:
        if geom is None or geom.is_empty: continue
        r = run_pipeline(geom, res)
        if r and AR_MIN <= r[1] <= AR_MAX:
            gt_angles.append(r[0])
gt_angles = np.array(gt_angles)
print(f"GT elongated boulders: {len(gt_angles)}")

In [ ]:
# Load pre-NMS shapefiles and compute orientations once per model.
# This is the slow step — orientations don't depend on NMS threshold.

model_data = {}   # name → {"gdf": ..., "angles": np.array, "ars": np.array}

for name, (pred_dir, glob) in MODELS.items():
    # Use pre-NMS file (no -nms suffix), falling back to post-NMS if needed
    shps = sorted(pred_dir.glob(glob))
    # Exclude the -nms files
    shps = [p for p in shps if "-nms" not in p.stem]
    if not shps:
        print(f"[{name}] Pre-NMS shapefile not found in {pred_dir} — skipping")
        continue

    print(f"\nLoading {name} pre-NMS ({shps[0].name})...")
    gdfs = [gpd.read_file(p) for p in shps]
    gdf  = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True), crs=gdfs[0].crs)
    gdf["poly_area"] = gdf.geometry.area
    gdf = gdf[gdf["poly_area"] >= AREAL_THRESHOLD].reset_index(drop=True)
    if "score" not in gdf.columns:
        gdf["score"] = 1.0
    print(f"  {len(gdf)} pre-NMS detections (after area filter)")

    # Compute orientations for every detection
    angles, ars = [], []
    for geom in tqdm(gdf.geometry, desc=f"  orientations", leave=False):
        r = run_pipeline(geom, res)
        if r:
            angles.append(r[0]); ars.append(r[1])
        else:
            angles.append(np.nan); ars.append(np.nan)

    gdf["angle180"]   = angles
    gdf["aspect_ratio"] = ars
    model_data[name] = gdf
    print(f"  orientations done")

In [ ]:
# Apply NMS at each threshold and record counts + elongated-boulder angles
sweep = {}   # name → {thresh → {"n_total": int, "angles": np.array}}

for name, gdf in model_data.items():
    sweep[name] = {}
    for thresh in NMS_THRESHOLDS:
        keep_mask = apply_nms_to_gdf(gdf, thresh)
        gdf_keep  = gdf[keep_mask]
        elongated = gdf_keep[
            (gdf_keep["aspect_ratio"] >= AR_MIN) &
            (gdf_keep["aspect_ratio"] <= AR_MAX)
        ]
        sweep[name][thresh] = {
            "n_total":  int(keep_mask.sum()),
            "angles":   elongated["angle180"].dropna().values,
        }
    print(f"{name}: " + "  ".join(
        f"NMS={t:.1f}→{sweep[name][t]['n_total']}" for t in NMS_THRESHOLDS))

In [ ]:
# Figure: detection count vs NMS threshold
COLORS = {
    "YOLOv8":          "#4C72B0",
    "SAM2 zero-shot":  "tomato",
    "SAM2 fine-tuned": "darkorange",
    "SAM2-auto":       "forestgreen",
}

fig, ax = plt.subplots(figsize=(7, 3.5))
for name, by_thresh in sweep.items():
    counts = [by_thresh[t]["n_total"] for t in NMS_THRESHOLDS]
    ax.plot(NMS_THRESHOLDS, counts, "o-", label=name,
            color=COLORS.get(name, "gray"), lw=1.5, ms=5)

ax.set_xlabel("NMS IoU threshold")
ax.set_ylabel("Detections (post-NMS)")
ax.set_title("Detection count vs. NMS threshold\n"
             "(SAM2 masks are larger → more sensitive to threshold than YOLO boxes)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
ax.set_xticks(NMS_THRESHOLDS)

# Mark the two key thresholds
ax.axvline(0.2, color="k", ls="--", lw=0.8, alpha=0.6, label="0.2 (YOLO original)")
ax.axvline(0.5, color="k", ls=":",  lw=0.8, alpha=0.6, label="0.5 (pre-fix SAM2)")

fig.tight_layout()
fig.savefig(OUT_DIR / "nms_counts.pdf", bbox_inches="tight")
plt.show()
print("Saved nms_counts.pdf")

In [ ]:
# Figure: orientation histograms — rows = thresholds, columns = models
model_names = list(sweep.keys())
n_thresh = len(NMS_THRESHOLDS)
n_models = len(model_names)

fig, axes = plt.subplots(n_thresh, n_models,
                          figsize=(2.6 * n_models, 2.0 * n_thresh),
                          sharex=True)

for row, thresh in enumerate(NMS_THRESHOLDS):
    for col, name in enumerate(model_names):
        ax     = axes[row, col]
        angles = sweep[name][thresh]["angles"]
        n      = sweep[name][thresh]["n_total"]
        color  = COLORS.get(name, "gray")

        counts, _ = np.histogram(angles, bins=BINS)
        cx = (BINS[:-1] + BINS[1:]) / 2
        ax.bar(cx, counts, width=4.5, color=color, edgecolor="white", lw=0.2)

        # GT overlay (scaled to match count)
        if len(angles) > 0:
            gc, _ = np.histogram(gt_angles, bins=BINS)
            ax.step(cx, gc * len(angles) / max(len(gt_angles), 1),
                    where="mid", color="seagreen", lw=0.9, alpha=0.8)
            ax.axhline(len(angles) / len(cx), color="k", ls="--", lw=0.5, alpha=0.4)

        ax.set_xlim(0, 180)
        ax.set_xticks([0, 90, 180])
        ax.spines[["top", "right"]].set_visible(False)

        if row == 0:
            ax.set_title(name, fontsize=8)
        if col == 0:
            ax.set_ylabel(f"NMS={thresh}\n(n={n})", fontsize=7)
        if row == n_thresh - 1:
            ax.set_xlabel("angle180 (°)", fontsize=7)

fig.suptitle(
    "Orientation histograms across NMS thresholds\n"
    "Green step = GT (scaled). Dashed = uniform. "
    "Lower threshold → more detections → choose threshold where SAM2 ≈ YOLO count.",
    fontsize=8, y=1.01)
fig.tight_layout()
fig.savefig(OUT_DIR / "nms_histograms.pdf", bbox_inches="tight")
plt.show()
print("Saved nms_histograms.pdf")